In [3]:
import math
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import os

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'Times New Roman'


In [4]:
# 配置与路径
N_GROUPS = 6
INCIDENCE_CSV = r"D:\app\OneDrive\Desktop\04 百日咳新策略\data\6group.csv"
POP_CSV = r"D:\app\OneDrive\Desktop\04 百日咳新策略\data\02_age_group.csv"
EXCEL_BD = r"D:\app\OneDrive\Desktop\04 百日咳新策略\data\chushengsiwang.xlsx"
years = list(range(2010, 2025))        # 2010-2024，对应你的 Excel
n_months_total = len(years) * 12      
OUT_DIR = Path("fit_segments")  # 输出目录
GROUP_LABELS = [f"G{i}" for i in range(1, N_GROUPS + 1)]

# 分段配置
SEGMENT_LENGTH = 96    # 每段长度（月）
SEGMENT_OVERLAP = 0    # 段间重叠（月，设为 0 表示不重叠）

# 模型参数（按月）
OMEGA = 30 / 10   # E->I
GAMMA = 30 / 35   # I->R
EPS1  = 1.0 / 60.0  # V->S
EPS2  = 1.0 / 120.0  # R->S
ALPHA = np.array([1/6.0, 1/18.0, 1/48.0, 1/48.0, 1/120.0, 0])  # aging rates
PHO   = np.array([0.99, 0.73, 0.0, 0.0, 0.0, 0])
V_EFF = np.array([0.90, 0.90, 0.85, 0, 0, 0.0])

# 季节性（默认；可按段覆盖）
SEASON_AMPL = 0.20   # 全局默认振幅（例如 0.10 表示 ±10%）
SEASON_PHASE = 0  # 全局默认相位（年为单位，0 表示 1 月为基线）

# 可选：为每一段定义振幅（长度应与 segments 对齐）。
# 留空或设为 None 使用全局默认。
SEASON_AMPL_PER_SEG = None  

# 基线 beta 矩阵（作为初始猜测）
BETA_BASE = np.array([
    [0.70, 0.50, 0.55, 0.10, 0.12, 0.5],
    [0.40, 0.35, 0.20, 0.15, 0.18, 0.5],
    [0.25, 0.20, 0.30, 0.18, 0.20, 0.3],
    [0.20, 0.15, 0.18, 0.18, 0.26, 0.1],
    [0.20, 0.15, 0.18, 0.18, 0.26, 0.1],
    [0.10, 0.10, 0.16, 0.14, 0.30, 0.1]
], dtype=float)


In [5]:
POP_COLUMN_MAP = {
    "0_6m": "G1",
    "6m_2y": "G2",
    "2_6y": "G3",
    "6_10y": "G4",
    "10_20y": "G5",
    "20_plus": "G6"
}


def load_incidence(csv_path: str) -> pd.DataFrame:
    """读取月度发病数：要求包含 group1..groupN 列；自动忽略时间列。"""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    for time_col in ("time", "t", "month"):
        if time_col in df.columns:
            df = df.drop(columns=[time_col])
    group_cols = [f"group{i}" for i in range(1, N_GROUPS + 1)]
    missing = [c for c in group_cols if c not in df.columns]
    if missing:
        raise ValueError(f"发病数据缺少列: {missing}")
    sub = df[group_cols].copy()
    for c in sub.columns:
        if sub[c].dtype == object:
            sub[c] = sub[c].astype(str).str.replace(",", "").str.replace(" ", "")
        sub[c] = pd.to_numeric(sub[c], errors='coerce').fillna(0.0)
    sub.columns = GROUP_LABELS
    return sub


def load_population(csv_path: str, n_months: int) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df.sort_values("year")
    rename_map = {src: dst for src, dst in POP_COLUMN_MAP.items() if src in df.columns}
    df = df.rename(columns=rename_map)
    missing = [g for g in GROUP_LABELS if g not in df.columns]
    if missing:
        raise ValueError(f"人口数据缺少列: {missing}")
    df[GROUP_LABELS] = df[GROUP_LABELS] * 10000.0

    monthly_rows = []
    for _, row in df.iterrows():
        monthly_rows.extend([row[GROUP_LABELS].values] * 12)

    pop_monthly = np.vstack(monthly_rows)
    if pop_monthly.shape[0] < n_months:
        pad = np.tile(pop_monthly[-1], (n_months - pop_monthly.shape[0], 1))
        pop_monthly = np.vstack([pop_monthly, pad])
    pop_monthly = pop_monthly[:n_months]
    return pd.DataFrame(pop_monthly, columns=GROUP_LABELS)


In [6]:
#出生人数和死亡率
def derive_births_and_deaths(xlsx_path: str, years: list, n_months: int, sheet_name: str = "Sheet3"):
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    df.columns = [c.strip().lower() for c in df.columns]
    birth_candidates = [c for c in df.columns if "birth" in c]
    birth_col = birth_candidates[0] if birth_candidates else "birth"
    death_cols = sorted([c for c in df.columns if c.startswith("death")])
    death_cols = death_cols[:N_GROUPS]
    df = df[["year", birth_col] + death_cols].copy()
    df = df.sort_values("year").reset_index(drop=True)
    births_monthly = []
    dr_monthly = []
    for _, row in df.iterrows():
        monthly_birth = row[birth_col] * 10000 / 12.0
        monthly_dr = np.array([row[c] / 12.0 for c in death_cols], dtype=float)
        births_monthly.extend([monthly_birth] * 12)
        dr_monthly.extend([monthly_dr] * 12)
    births_monthly = np.array(births_monthly[:n_months])
    dr_monthly = np.array(dr_monthly[:n_months])
    return births_monthly, dr_monthly

#模型方程
def vseir_rhs(t, y, params):
    y = np.maximum(y, 0.0)

    beta_flat = params["beta_flat"]
    births_series = params["births_monthly"]
    dr_series = params["dr_monthly"]
    season_ampl = params.get("season_ampl", SEASON_AMPL)
    season_phase = params.get("season_phase", SEASON_PHASE)
    n = params.get("n_groups", N_GROUPS)
    beta_mat = np.asarray(beta_flat).reshape(n, n)

    idx = int(np.floor(t))
    idx = np.clip(idx, 0, len(births_series) - 1)
    births_inflow = births_series[idx]
    dr = np.asarray(dr_series[idx], dtype=float)

    V = y[0:n]
    S = y[n:2*n]
    E = y[2*n:3*n]
    I = y[3*n:4*n]
    R = y[4*n:5*n]
    N = V + S + E + I + R
    infectious_frac = np.divide(I, N, out=np.zeros_like(I), where=N > 0)
    lam = beta_mat @ infectious_frac

    season_factor = 1.2 + season_ampl * np.sin(2.0 * np.pi * (t/12.0 + season_phase))
    season_factor = np.maximum(season_factor, 0.01)
    lam = np.maximum(lam * season_factor, 0.0)

    alpha_vec = ALPHA[:n]
    dS = EPS2 * R + EPS1 * V - lam * S - (dr + alpha_vec) * S
    dE = lam * S - OMEGA * E - (dr + alpha_vec) * E
    dI = OMEGA * E - GAMMA * I - (dr + alpha_vec) * I
    dR = GAMMA * I - EPS2 * R - (dr + alpha_vec) * R
    dV = -EPS1 * V - (dr + alpha_vec) * V
    dS[0] += births_inflow

    for g in range(1, n):
        gm1 = g - 1
        ag_s_out = ALPHA[gm1] * S[gm1]
        to_V = PHO[gm1] * V_EFF[gm1] * ag_s_out
        to_S = (1.0 - PHO[gm1] * V_EFF[gm1]) * ag_s_out
        dV[g] += to_V + ALPHA[gm1] * V[gm1]
        dS[g] += to_S
        dE[g] += ALPHA[gm1] * E[gm1]
        dI[g] += ALPHA[gm1] * I[gm1]
        dR[g] += ALPHA[gm1] * R[gm1]

    return np.concatenate([dV, dS, dE, dI, dR])


def simulate(times, y0, params):
    y0 = np.maximum(y0, 0.0)

    def rhs_with_clipping(t, y):
        dy = vseir_rhs(t, y, params)
        y_clipped = np.maximum(y, 0.0)
        for i in range(len(y)):
            if y_clipped[i] < 1e-10 and dy[i] < 0:
                dy[i] = max(dy[i], -y_clipped[i] * 0.9)
        return dy

    sol = solve_ivp(rhs_with_clipping, (times[0], times[-1]), y0,
                    t_eval=times, method='RK45', vectorized=False, rtol=1e-6, atol=1e-9)

    if not sol.success:
        raise RuntimeError(sol.message)

    Y = np.maximum(sol.y, 0.0)
    if np.any(Y < -1e-10):
        print(f"警告：检测到负的状态变量，已强制为非负。最小值: {np.min(Y)}")
        Y = np.maximum(Y, 0.0)

    return Y


def model_curve_from_state(state_y):
    n = N_GROUPS
    E = np.maximum(state_y[2*n:3*n, :], 0.0)
    return OMEGA * E


def fit_beta_full(times, y0, incidence_obs, births_series, dr_series,
                  weight_power: float = 1.5, weight_eps: float = 10.0,
                  season_ampl: float = None, season_phase: float = None,
                  beta_init: np.ndarray = None,
                  enable_reporting: bool = True,
                  rho_init: np.ndarray = None,
                  rho_prev: np.ndarray = None,
                  rho_smooth_lambda: float = 0.0,
                  rho_bounds=(0.01, 1.0),
                  rho_prior_alpha: float = 6.0,
                  rho_prior_beta: float = 2.0,
                  rho_prior_weight: float = 1.0):

    n = N_GROUPS
    beta_dim = n * n

    if beta_init is not None:
        x0_beta = np.maximum(beta_init.flatten(), 0.05)
    else:
        x0_beta = np.maximum(BETA_BASE.flatten(), 0.05)

    lb_beta = np.full(beta_dim, 1e-7)
    ub_beta = np.full(beta_dim, 2.0)

    if enable_reporting:
        if rho_init is None:
            rho_init = np.full(n, 0.9)
        rho_init = np.clip(np.asarray(rho_init, dtype=float), rho_bounds[0], rho_bounds[1])
        if rho_prev is not None:
            rho_prev_vec = np.clip(np.asarray(rho_prev, dtype=float), rho_bounds[0], rho_bounds[1])
        else:
            rho_prev_vec = None
        x0 = np.concatenate([x0_beta, rho_init])
        lb = np.concatenate([lb_beta, np.full(n, rho_bounds[0])])
        ub = np.concatenate([ub_beta, np.full(n, rho_bounds[1])])
    else:
        x0 = x0_beta
        lb = lb_beta
        ub = ub_beta
        rho_prev_vec = None

    group_means = np.maximum(incidence_obs.mean(axis=0), 0.0)
    weights = np.power(group_means + weight_eps, weight_power)
    weights = weights / np.mean(weights)

    def unpack_params(x):
        beta_flat = x[:beta_dim]
        if enable_reporting:
            rho = x[beta_dim:beta_dim + n]
        else:
            rho = np.ones(n)
        return beta_flat, rho

    def residuals(x):
        beta_flat, rho = unpack_params(x)
        params = {
            "beta_flat": beta_flat,
            "births_monthly": births_series,
            "dr_monthly": dr_series,
            "season_ampl": season_ampl if season_ampl is not None else SEASON_AMPL,
            "season_phase": season_phase if season_phase is not None else SEASON_PHASE,
            "n_groups": n,
        }
        Y = simulate(times, y0, params)
        curve = model_curve_from_state(Y)
        pred = rho[:, None] * curve
        pred = np.maximum(pred, 0.0)
        diff = (pred - incidence_obs.T)
        res = (weights[:, None] * diff).ravel(order='F')

        if enable_reporting and rho_prior_weight > 0:
            rho_clipped = np.clip(rho, rho_bounds[0] + 1e-6, rho_bounds[1] - 1e-6)
            penalties = []
            for g in range(n):
                rho_g = rho_clipped[g]
                log_prior = -(rho_prior_alpha - 1.0) * np.log(rho_g) - (rho_prior_beta - 1.0) * np.log(1.0 - rho_g)
                log_prior = max(log_prior, 0.0)
                penalties.append(np.sqrt(2.0 * rho_prior_weight * log_prior))
            res = np.concatenate([res, np.array(penalties)])

        if enable_reporting and rho_prev_vec is not None and rho_smooth_lambda > 0:
            smooth_penalty = np.sqrt(rho_smooth_lambda) * (rho - rho_prev_vec)
            res = np.concatenate([res, smooth_penalty])

        return res

    res = least_squares(residuals, x0, bounds=(lb, ub), max_nfev=500, verbose=0)
    beta_opt, rho_opt = unpack_params(res.x)
    return beta_opt, rho_opt


In [7]:
# 加载数据
inc_df = load_incidence(INCIDENCE_CSV)
n_months = inc_df.shape[0]
pop_df = load_population(POP_CSV, n_months)

# 创建输出目录
OUT_DIR.mkdir(exist_ok=True)
segments = [(0, 96), (95, 121), (120, 166), (165, 180), (179, 192)]


In [8]:
# 定义单段拟合与保存函数；按需逐段在独立 cell 调用
results_summary = []

# 预先计算整段的出生与死亡月序列（只需一次）
births_full, dr_full = derive_births_and_deaths(EXCEL_BD, years, n_months, sheet_name="Sheet3")


def _build_initial_state(pop_segment: pd.DataFrame, inc_segment: pd.DataFrame):
    N0 = pop_segment.iloc[0].values.astype(float)
    vacc_heuristics = [0.0, 0.9, 0.7, 0.5, 0.3,0.2]
    V0 = np.zeros(N_GROUPS)
    for idx in range(N_GROUPS):
        if idx < len(vacc_heuristics):
            V0[idx] = vacc_heuristics[idx] * N0[idx]
    k = max(1, min(3, len(inc_segment)))
    cases0 = inc_segment.iloc[:k].mean(axis=0).values.astype(float)
    I0 = cases0 / max(GAMMA, 1e-6)
    E0 = cases0 / max(OMEGA, 1e-6)
    R0 = np.zeros(N_GROUPS)
    if N_GROUPS > 1:
        R0[1] = 30
    if N_GROUPS > 2:
        R0[2] = 0.1 * N0[2]
    if N_GROUPS > 3:
        R0[3] = 0.4 * N0[3]
    if N_GROUPS > 4:
        R0[4] = 0.4 * N0[4]
    if N_GROUPS > 5:
        R0[4] = 0.4 * N0[4]
    S0 = np.maximum(N0 - V0 - E0 - I0 - R0, 0.0)
    return np.concatenate([V0, S0, E0, I0, R0])


def run_segment(seg_idx: int, prev_y_end, accept: bool = True, season_ampl: float = None,
                season_phase: float = None, weight_power: float = None, weight_eps: float = None, 
                prev_beta: np.ndarray = None, prev_rho: np.ndarray = None,
                rho_smooth_lambda: float = 0.0, rho_bounds: tuple = None):
    seg_start, seg_end = segments[seg_idx]
    print(f"\n{'='*60}")
    print(f"处理段 {seg_idx+1}/{len(segments)}: 月份 {seg_start} 到 {seg_end-1}")
    print(f"{'='*60}")

    inc_seg = inc_df.iloc[seg_start:seg_end].copy()
    pop_seg = pop_df.iloc[seg_start:seg_end].copy()
    seg_len = seg_end - seg_start

    if season_ampl is not None:
        seg_season_ampl = float(season_ampl)
    elif SEASON_AMPL_PER_SEG is not None:
        assert len(SEASON_AMPL_PER_SEG) == len(segments), "SEASON_AMPL_PER_SEG 长度需与 segments 一致"
        seg_season_ampl = float(SEASON_AMPL_PER_SEG[seg_idx])
    else:
        seg_season_ampl = SEASON_AMPL
    
    if season_phase is not None:
        seg_season_phase = float(season_phase)
    else:
        seg_season_phase = SEASON_PHASE

    times = np.arange(seg_start, seg_end, dtype=float)

    if prev_y_end is None:
        y0 = _build_initial_state(pop_seg, inc_seg)
    else:
        y0 = np.maximum(prev_y_end, 0.0)

    inc_obs = inc_seg.values.astype(float)

    print("开始拟合...")
    fit_kwargs = {
        "season_ampl": seg_season_ampl,
        "season_phase": seg_season_phase,
    }
    if weight_power is not None:
        fit_kwargs["weight_power"] = weight_power
    if weight_eps is not None:
        fit_kwargs["weight_eps"] = weight_eps
    if prev_beta is not None:
        fit_kwargs["beta_init"] = prev_beta
    if prev_rho is not None:
        fit_kwargs["rho_init"] = prev_rho
        fit_kwargs["rho_prev"] = prev_rho
    if rho_smooth_lambda is not None:
        fit_kwargs["rho_smooth_lambda"] = float(rho_smooth_lambda)
    if rho_bounds is not None:
        fit_kwargs["rho_bounds"] = rho_bounds

    beta_flat, reporting_rates = fit_beta_full(
        times, y0, inc_obs, births_full, dr_full, **fit_kwargs
    )
    beta_mat = beta_flat.reshape(N_GROUPS, N_GROUPS)
    print("拟合完成")
    print(f"报告率（每组）: {np.round(reporting_rates, 6)}")

    params = {
        "beta_flat": beta_flat,
        "births_monthly": births_full,
        "dr_monthly": dr_full,
        "season_ampl": seg_season_ampl,
        "season_phase": seg_season_phase,
        "n_groups": N_GROUPS,
    }
    Y = simulate(times, y0, params)
    curve_flow = model_curve_from_state(Y)
    curve_fit = reporting_rates[:, None] * curve_flow
    end_state = Y[:, -1]

    seg_dir = OUT_DIR / f"segment_{seg_idx+1:02d}_months_{seg_start:04d}_{seg_end-1:04d}"
    seg_dir.mkdir(exist_ok=True)

    beta_df = pd.DataFrame(beta_mat, index=[f"Group{i}" for i in range(1, N_GROUPS+1)],
                          columns=[f"Group{i}" for i in range(1, N_GROUPS+1)])
    beta_df.to_csv(seg_dir / "beta_matrix.csv")

    reporting_series = pd.Series(reporting_rates, index=[f"Group{i}" for i in range(1, N_GROUPS+1)])
    reporting_series.to_csv(seg_dir / "reporting_rates.csv")

    comp_labels = [f"{comp}{i}" for comp in ("V", "S", "E", "I", "R") for i in range(1, N_GROUPS+1)]
    pd.Series(y0, index=comp_labels).to_csv(seg_dir / "initial_state.csv")
    pd.Series(end_state, index=comp_labels).to_csv(seg_dir / "final_state.csv")
    states_df = pd.DataFrame(Y.T, columns=comp_labels)
    states_df.to_csv(seg_dir / "states_timeseries.csv", index=False)

    for g in range(N_GROUPS):
        plt.figure(figsize=(6,4))
        plt.plot(inc_obs[:, g], label='Actual (cases)', color='black', linewidth=1.5)
        plt.plot(curve_fit[g, :], label='Fitted (reported)', color='tab:blue', linewidth=1.5)
        plt.title(f'Segment {seg_idx+1} - Group G{g+1} (months {seg_start}-{seg_end-1})')
        plt.xlabel('Month')
        plt.ylabel('Monthly onsets (reported)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(seg_dir / f"group_{g+1}.png", dpi=150)
        plt.close()

    results_summary.append({
        'segment': seg_idx + 1,
        'start_month': seg_start,
        'end_month': seg_end - 1,
        'length': seg_len,
        'beta_matrix': beta_mat,
        'output_dir': str(seg_dir),
        'season_ampl': seg_season_ampl,
        'reporting_rates': reporting_rates,
    })

    print(f"结果已保存到: {seg_dir}")

    next_prev = end_state if accept else prev_y_end
    if accept:
        print("已接受：下一段将使用该段末状态作为初值。")
    else:
        print("未接受：下一段将沿用传入的初值（不更新末状态）。")
    return next_prev, end_state, beta_mat, reporting_rates

print("已定义 run_segment(seg_idx, prev_y_end, accept=True, season_ampl=None)。在下面的 cell 中逐段调用。")


已定义 run_segment(seg_idx, prev_y_end, accept=True, season_ampl=None)。在下面的 cell 中逐段调用。


In [9]:
# 段 1（0-based: seg_idx=0）
# 按需修改 accept=True/False；如需单独设定该段季节振幅，传入 season_ampl=0.2 之类
prev_y_end_seg1 = None
prev_y_end_seg1, end_state_1, beta_1, rho_1 = run_segment(
    seg_idx=0,
    prev_y_end=prev_y_end_seg1,
    accept=True,
    season_ampl=None,
    rho_smooth_lambda=0.0
)



处理段 1/5: 月份 0 到 95
开始拟合...
拟合完成
报告率（每组）: [0.958948 0.863799 0.862475 0.833883 0.833535 0.833698]
结果已保存到: fit_segments\segment_01_months_0000_0095
已接受：下一段将使用该段末状态作为初值。


In [10]:
# 段 2（seg_idx=1）
# 如果上一段不接受，则把 accept=False 放在上一个 cell；这里会沿用未更新的 prev_y_end
# 使用独立的变量名，从段1的末尾值初始化
prev_y_end_seg2 = prev_y_end_seg1  # 从段1的末尾值初始化
prev_y_end_seg2, end_state_2, beta_2, rho_2 = run_segment(
    seg_idx=1,
    prev_y_end=prev_y_end_seg2,
    accept=True,
    season_ampl=0.8,
    season_phase=1/12,
    weight_eps=8,
    prev_beta=None,  # 使用前一段的beta作为初始值
    prev_rho=None,
    rho_smooth_lambda=0
)




处理段 2/5: 月份 95 到 120
开始拟合...
拟合完成
报告率（每组）: [0.999932 0.999979 0.971954 0.784425 0.371756 0.172672]
结果已保存到: fit_segments\segment_02_months_0095_0120
已接受：下一段将使用该段末状态作为初值。


In [11]:
# 段 3（seg_idx=2）
prev_y_end_seg3 = prev_y_end_seg2  # 从段2的末尾值初始化
prev_y_end_seg3, end_state_3, beta_3, rho_3 = run_segment(
    seg_idx=2,
    prev_y_end=prev_y_end_seg3,
    accept=True,
    season_ampl=1.0,  # 增加季节性振幅以更好捕捉暴发
    season_phase=-1/12,
    weight_power=1.3,
    weight_eps=10,
    prev_beta=beta_2,  # 使用前一段的beta作为初始值
    prev_rho=rho_2,
    rho_smooth_lambda=5
)


处理段 3/5: 月份 120 到 165
开始拟合...
拟合完成
报告率（每组）: [0.833188 0.491118 0.396486 0.07593  0.01     0.010967]
结果已保存到: fit_segments\segment_03_months_0120_0165
已接受：下一段将使用该段末状态作为初值。


In [12]:
# 段 4（seg_idx=3）- 针对大暴发优化参数
# 使用独立的变量名，从段3的末尾值初始化
prev_y_end_seg4 = prev_y_end_seg3  # 从段3的末尾值初始化
prev_y_end_seg4, end_state_4, beta_4, rho_4 = run_segment(
    seg_idx=3,
    prev_y_end=prev_y_end_seg4,
    accept=True,
    season_ampl=1.0,  # 增加季节性振幅以更好捕捉暴发
    season_phase=2/12,  # 调整季节相位
    weight_power=1.0,  # 降低权重指数，避免过度偏重高发组
    weight_eps=10,   # 增加低值保护，确保低发病组也能被拟合
    prev_beta=beta_3, 
    prev_rho=rho_3,
    rho_smooth_lambda=10
)



处理段 4/5: 月份 165 到 179
开始拟合...
拟合完成
报告率（每组）: [0.207071 0.707598 0.652666 0.184266 0.054852 0.013461]
结果已保存到: fit_segments\segment_04_months_0165_0179
已接受：下一段将使用该段末状态作为初值。


In [31]:
# 段 5（seg_idx=4）修正版：调整疫苗接种率 PHO 第3组参数
# 如需更新覆盖率，请修改下方 PHO[2] 的值，再运行本单元
PHO_backup = PHO.copy()
PHO[2] = 0.7  # 覆盖率，索引2对应第3年龄组

prev_y_end_seg5 = prev_y_end_seg4  # 依段4末状态作为初值
prev_y_end_seg5, end_state_5, beta_5, rho_5 = run_segment(
    seg_idx=4,
    prev_y_end=prev_y_end_seg5,
    accept=True,
    season_ampl=0.7,
    season_phase=3/12,
    weight_power=0.7,
    weight_eps=1,
    prev_beta=beta_4,
    prev_rho=rho_4,
    rho_smooth_lambda=5
)


处理段 5/5: 月份 179 到 191
开始拟合...
拟合完成
报告率（每组）: [0.051939 0.203839 0.095922 0.022321 0.01     0.011343]
结果已保存到: fit_segments\segment_05_months_0179_0191
已接受：下一段将使用该段末状态作为初值。
